# Week 6 lecture walkthrough: diagram summaries, distances and validation

This is the live worked example, built around a noisy circle, a small perturbation and a coordinate-shuffled surrogate. It follows the conceptual argument of the slides through prediction, reveal and interpretation. It is not intended as a line-by-line answer key to the participant practical.

**Resource boundary.** The [reference notes](index.qmd) compare summaries and validation claims. The [slides](slides.qmd) ask what each representation forgets. This notebook supplies the completed matching, summaries and surrogate comparison. The [participant practical](lab.ipynb) requires students to select and defend a comparison.

**Lecture map.** Work one diagram matching by hand, predict the ordering of distances, then compare direct, functional and vector summaries. Finish by placing one statistic against an explicit surrogate rather than interpreting its magnitude alone.

Diagrams are not conclusions. This laboratory compares direct diagram distances, Betti curves, vector summaries and a surrogate null. Every summary forgets something. All examples use $H_1$ over $\mathbb F_2$.

**Presenter route.** Keep the base cloud fixed. Reveal how bottleneck, Wasserstein, Betti curves, images and a maximum-persistence null each forget different information.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
RNG=np.random.default_rng(3024)
from ripser import ripser
from persim import bottleneck
from scipy.optimize import linear_sum_assignment

def betti_curve(D,grid):
    return np.array([np.sum((D[:,0]<=a)&(a<D[:,1])) for a in grid])

def persistence_image(D,bounds=(0,2,0,2),n=24,sigma=.09):
    finite=D[np.isfinite(D[:,1])]
    xs=np.linspace(bounds[0],bounds[1],n); ys=np.linspace(bounds[2],bounds[3],n)
    X,Y=np.meshgrid(xs,ys); img=np.zeros_like(X)
    for b,d in finite:
        persistence=d-b
        img+=persistence*np.exp(-((X-b)**2+(Y-persistence)**2)/(2*sigma**2))
    return img

def finite_lengths(D):
    F=D[np.isfinite(D[:,1])]
    return F[:,1]-F[:,0] if len(F) else np.array([])

def max_persistence(D):
    lengths=finite_lengths(D); return float(np.max(lengths)) if len(lengths) else 0.

def total_persistence(D,q=1):
    return float(np.sum(finite_lengths(D)**q))

def persistence_entropy(D):
    lengths=finite_lengths(D); total=lengths.sum()
    if total<=0: return 0.
    probabilities=lengths/total
    return float(-np.sum(probabilities*np.log(probabilities)))

def wasserstein_linf(D,E,q=1):
    D=D[np.all(np.isfinite(D),axis=1)]; E=E[np.all(np.isfinite(E),axis=1)]
    n,m=len(D),len(E); C=np.full((n+m,n+m),np.inf)
    if n and m: C[:n,:m]=np.max(np.abs(D[:,None,:]-E[None,:,:]),axis=2)**q
    for i,(b,d) in enumerate(D): C[i,m+i]=((d-b)/2)**q
    for j,(b,d) in enumerate(E): C[n+j,j]=((d-b)/2)**q
    C[n:,m:]=0
    rows,cols=linear_sum_assignment(C)
    return float(C[rows,cols].sum()**(1/q))

## Hand checkpoint: why the diagonal is available

Let $D=\{(0,2),(1,1.4)\}$ and $E=\{(0.1,2.1)\}$. Match the long points and send the short point to the diagonal.

Predict the $L_\infty$ cost of each match. The distance from $(b,d)$ to the diagonal is $(d-b)/2$.

In [ ]:
long_match_cost = max(abs(0.0 - 0.1), abs(2.0 - 2.1))
short_to_diagonal = (1.4 - 1.0) / 2
candidate_bottleneck = max(long_match_cost, short_to_diagonal)
print('long match:', long_match_cost)
print('short point to diagonal:', short_to_diagonal)
print('candidate bottleneck cost:', candidate_bottleneck)
assert np.isclose(candidate_bottleneck, 0.2)

## 1. Observe: lecture demonstration

**Presenter cue.** Show the object before the calculation. Ask the room to separate what is given from what will be constructed.

Create a noisy circle, a slightly perturbed copy, and a coordinate-shuffled surrogate that preserves the separate $x$ and $y$ marginals but destroys their pairing.

In [ ]:
n=80; theta=np.linspace(0,2*np.pi,n,endpoint=False)
base=np.c_[np.cos(theta),np.sin(theta)]+.05*RNG.normal(size=(n,2))
perturbed=base+.025*RNG.normal(size=base.shape)
surrogate=base.copy(); surrogate[:,1]=RNG.permutation(surrogate[:,1])
datasets={'base':base,'perturbed':perturbed,'surrogate':surrogate}
diagrams={k:ripser(P,maxdim=1)['dgms'][1] for k,P in datasets.items()}
fig,axes=plt.subplots(1,3,figsize=(9,3))
for ax,(name,P) in zip(axes,datasets.items()): ax.scatter(*P.T,s=12); ax.set_title(name); ax.set_aspect('equal')
plt.show()

## 2. Predict: lecture demonstration

**Presenter cue.** Pause here and collect at least two predictions before revealing any output.

1. Which pair should have the smallest bottleneck distance?
2. When might Wasserstein distance react more strongly than bottleneck distance?
3. Can two diagrams share maximum persistence but differ substantially elsewhere?
4. What property does the shuffled surrogate preserve, and what structure does it destroy?

## 3. Implement: lecture demonstration

**Reveal.** Run one cell at a time. Name the domain, codomain, complex, module or summary before interpreting its values.

### A. Direct distances

Bottleneck records the worst matched cost. The supplied Wasserstein calculation uses the same $L_\infty$ ground cost and accumulates order-$q$ matching costs. Both allow matching to the diagonal.

In [ ]:
for other in ['perturbed','surrogate']:
 print(other,'bottleneck',round(bottleneck(diagrams['base'],diagrams[other]),3),
       'Wasserstein q=1, L_inf',round(wasserstein_linf(diagrams['base'],diagrams[other],q=1),3))

### B. Scalars, Betti curves and persistence images

Maximum persistence, total persistence and persistence entropy reduce the diagram to different single numbers. A Betti curve counts intervals alive at each scale. The image below smooths birth-death points onto a fixed grid, weighted by persistence.

In [ ]:
for name,D in diagrams.items():
 print(name,'maximum',round(max_persistence(D),3),
       'total',round(total_persistence(D),3),
       'entropy',round(persistence_entropy(D),3))

grid=np.linspace(0,2,150)
fig,axes=plt.subplots(1,2,figsize=(10,3.5))
for name,D in diagrams.items(): axes[0].plot(grid,betti_curve(D,grid),label=name)
axes[0].set_xlabel('distance threshold'); axes[0].set_ylabel('beta_1'); axes[0].legend()
axes[1].imshow(persistence_image(diagrams['base']),origin='lower',extent=[0,2,0,2]); axes[1].set_title('base persistence image')
axes[1].set_xlabel('birth'); axes[1].set_ylabel('persistence')
plt.show()

## 4. Compare: lecture demonstration

**Controlled comparison.** Keep the stated input fixed and change only the highlighted modelling decision.

Generate 40 coordinate-shuffled surrogates. Compare the observed maximum persistence with its empirical null distribution. This is a demonstration of a null pattern, not a calibrated scientific test.

In [ ]:
null=[]
for _ in range(40):
 P=base.copy(); P[:,1]=RNG.permutation(P[:,1]); null.append(max_persistence(ripser(P,maxdim=1)['dgms'][1]))
observed=max_persistence(diagrams['base'])
print('observed',round(observed,3),'null 95% quantile',round(float(np.quantile(null,.95)),3),
      'empirical exceedance',(1+sum(v>=observed for v in null))/(1+len(null)))
plt.hist(null,bins=12); plt.axvline(observed,color='red'); plt.xlabel('maximum H1 persistence'); plt.show()

## 5. Interpret: lecture demonstration

**Presenter close.** Ask what the example supports, what information was discarded and which stronger claim would be unjustified.

1. Which summary emphasises one worst feature, and which accumulates many differences?
2. How do maximum persistence, total persistence and entropy answer different questions?
3. What information is lost by a Betti curve or persistence image?
4. What exchangeability or mechanism would justify the surrogate?
5. Why is an empirical exceedance not automatically a scientific p-value?
6. Which non-topological baseline should be evaluated beside the diagram summary?

**◇ Object check.** Diagram, distance, curve, image and scalar test statistic are successive summaries. None reconstructs the observed system.

The perturbed copy should remain close to the base. Bottleneck is controlled by the worst match, whereas Wasserstein can grow through many modest discrepancies. The surrogate test is meaningful only relative to a justified data-generating null; coordinate shuffling is pedagogical and discards dependence in a very particular way.

## Lecture close

Return to the final slide questions.

1. Name the observed or starting object.
2. Name every constructed object used in this walkthrough.
3. Identify the single modelling decision that drove the central comparison.
4. State one conclusion supported by the calculation and one conclusion it cannot establish.

**Take-forward example.** The purpose of a noisy circle, a small perturbation and a coordinate-shuffled surrogate is to make diagram summaries, distances and validation concrete. The example is deliberately small or synthetic so that the construction remains inspectable.

## Optional extension: chromatic information

These panels have identical point locations and therefore identical ordinary persistence. Their labels encode different organisation. Chromatic persistence retains the labels so that mixing, contribution and surrounding become available questions.

In [ ]:
angles = np.linspace(0, 2*np.pi, 96, endpoint=False)
radii = np.repeat([0.65, 1.0], 48)
theta = np.tile(angles[:48], 2)
label_cloud = np.c_[radii*np.cos(theta), radii*np.sin(theta)]
mixed = np.arange(96) % 2
separated = (radii > 0.8).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, labels, title in zip(axes, [mixed, separated], ['labels mixed', 'labels separated by ring']):
    ax.scatter(*label_cloud.T, c=np.where(labels, 'tab:blue', 'tab:red'), s=18)
    ax.set_title(title); ax.set_aspect('equal'); ax.axis('off')
plt.show()
ordinary_a = ripser(label_cloud, maxdim=1)['dgms'][1]
ordinary_b = ripser(label_cloud, maxdim=1)['dgms'][1]
print('ordinary diagrams identical:', np.allclose(ordinary_a, ordinary_b))